# 02.5 — Optimize and observe lab

1. Generation parameters, measured rather than described
2. What reasoning models ignore, proved by calling them
3. Prompt engineering: few-shot, chain of thought, decomposition
4. Reflection: self-critique, critic model, and a hard verifier
5. OpenTelemetry tracing to Application Insights
6. Token, latency, and safety analytics from your own instrumentation
7. Orchestrating models: routing, cascade, fallback, and a rules-first hybrid
8. Cleanup

**Cost:** this unit makes several calls per question in sections 4 and 7, so it
costs more than the others — still cents, not dollars. Section 5 writes a small
amount of telemetry into Application Insights, which is billed by ingestion volume.

In [ ]:
import sys, pathlib, json, time, statistics

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client, project_client, ask, show_usage

client = chat_client()
MINI = cfg["MODEL_MINI"]
BIG = cfg.get("MODEL_CHAT") or MINI
REASONING = cfg.get("MODEL_REASONING")

print("mini      :", MINI)
print("large     :", BIG)
print("reasoning :", REASONING or "(not deployed - some cells will skip)")

## 1. Generation parameters

Measure, do not assume. `distinct_ratio` below is the number of unique responses
over the number of samples — a crude but honest determinism metric.

In [ ]:
PROMPT = "Write a one-sentence product description for a rugged industrial controller."


def sample(n=4, model=None, **params):
    outs = []
    for _ in range(n):
        r = client.chat.completions.create(
            model=model or MINI,
            messages=[{"role": "user", "content": PROMPT}],
            max_tokens=60,
            **params,
        )
        outs.append((r.choices[0].message.content or "").strip())
    return outs


for label, params in [
    ("temperature=0.0", {"temperature": 0.0}),
    ("temperature=1.0", {"temperature": 1.0}),
    ("temperature=1.8", {"temperature": 1.8}),
    ("top_p=0.1     ", {"top_p": 0.1}),
    ("top_p=1.0     ", {"top_p": 1.0}),
]:
    outs = sample(**params)
    ratio = len(set(outs)) / len(outs)
    print(f"{label}  distinct_ratio={ratio:.2f}")
    print(f"    {outs[0][:110]}")

`temperature=0` and `top_p=0.1` both collapse variety — they are two ways of
narrowing the same distribution. **Tune one, leave the other at its default.**
Setting both is the classic mistake and produces unpredictable interactions.

Note also that `temperature=0` is *near*-deterministic, not deterministic. Batching
and floating-point non-determinism on the server can still produce different text.

In [ ]:
# seed: best-effort reproducibility, valid only while system_fingerprint is stable.
seeded = [
    client.chat.completions.create(
        model=MINI, messages=[{"role": "user", "content": PROMPT}],
        temperature=1.0, seed=42, max_tokens=60,
    ) for _ in range(3)
]
texts = [s.choices[0].message.content for s in seeded]
prints = [getattr(s, "system_fingerprint", None) for s in seeded]

print("identical outputs :", len(set(texts)) == 1)
print("fingerprints      :", set(prints))
print("\nIf the fingerprint changes between calls, the seed guarantee is void.")

In [ ]:
# stop and max_tokens. Note what finish_reason tells you, and that the stop string
# is NOT included in the output.
stopped = client.chat.completions.create(
    model=MINI,
    messages=[{"role": "user", "content": "List three colours, one per line, then write END."}],
    stop=["END"], temperature=0, max_tokens=60,
)
print("stop      ->", repr(stopped.choices[0].message.content))
print("finish    ->", stopped.choices[0].finish_reason)

truncated = client.chat.completions.create(
    model=MINI,
    messages=[{"role": "user", "content": "Explain vector databases in detail."}],
    max_tokens=20, temperature=0,
)
print("\ntruncated ->", repr(truncated.choices[0].message.content))
print("finish    ->", truncated.choices[0].finish_reason, "  <- users see a cut-off sentence")

In [ ]:
# frequency_penalty vs presence_penalty. Same direction, different trigger:
# frequency scales with HOW OFTEN a token appeared; presence fires on first repeat.
REPEAT = "Describe the benefits of cloud computing. Be repetitive and use the word 'scalable' often."

for label, kw in [
    ("no penalty      ", {}),
    ("frequency=1.5   ", {"frequency_penalty": 1.5}),
    ("presence=1.5    ", {"presence_penalty": 1.5}),
]:
    text = client.chat.completions.create(
        model=MINI, messages=[{"role": "user", "content": REPEAT}],
        temperature=0.7, max_tokens=110, **kw,
    ).choices[0].message.content
    words = text.lower().split()
    unique = len(set(words)) / max(len(words), 1)
    print(f"{label} 'scalable' x{words.count('scalable')}  lexical_diversity={unique:.2f}")

## 2. What reasoning models ignore

| Parameter | `gpt-4o-mini` | `o4-mini` |
|---|---|---|
| `temperature`, `top_p`, penalties | honoured | ignored or rejected |
| `logprobs` | yes | not supported |
| `max_tokens` | yes | **rejected** — use `max_completion_tokens` |
| `reasoning_effort` | n/a | `low` / `medium` / `high` |
| Hidden reasoning tokens billed | no | **yes** |

Prove it rather than trusting the table.

In [ ]:
if not REASONING:
    print("MODEL_REASONING not deployed - skipping section 2")
else:
    print("a) max_tokens on a reasoning model:")
    try:
        client.chat.completions.create(
            model=REASONING, messages=[{"role": "user", "content": "2+2?"}], max_tokens=50)
        print("   accepted (SDK or service may be translating it)")
    except Exception as e:
        print("  ", type(e).__name__, str(e)[:190])

    print("\nb) temperature on a reasoning model:")
    try:
        client.chat.completions.create(
            model=REASONING, messages=[{"role": "user", "content": "2+2?"}],
            temperature=0.9, max_completion_tokens=2000)
        print("   accepted but not honoured - reasoning models do not expose sampling controls")
    except Exception as e:
        print("  ", type(e).__name__, str(e)[:190])

In [ ]:
PUZZLE = (
    "A shipment holds 3 crates. A weighs twice B. C weighs 12kg less than A. "
    "Total 108kg. No crate may exceed 50kg. Give each weight and say LEGAL or ILLEGAL."
)


def measure(model, messages, **kw):
    t0 = time.perf_counter()
    r = client.chat.completions.create(model=model, messages=messages, **kw)
    dt = time.perf_counter() - t0
    u = r.usage
    details = getattr(u, "completion_tokens_details", None)
    hidden = getattr(details, "reasoning_tokens", 0) or 0 if details else 0
    visible = u.completion_tokens - hidden
    print(f"{model:<16} {dt:>5.1f}s  in={u.prompt_tokens:<5} visible={visible:<5} hidden={hidden:<5}")
    return r


msgs = [{"role": "user", "content": PUZZLE}]
measure(MINI, msgs, temperature=0, max_tokens=400)
measure(MINI, [{"role": "user", "content": PUZZLE + " Think step by step first."}],
        temperature=0, max_tokens=700)

if REASONING:
    for effort in ("low", "high"):
        try:
            measure(REASONING, msgs, max_completion_tokens=4000, reasoning_effort=effort)
        except Exception as e:
            print(f"reasoning_effort={effort} unsupported:", str(e)[:130])
            measure(REASONING, msgs, max_completion_tokens=4000)
            break

The `hidden` column is the whole argument. You pay for those tokens and never see
them, and `high` reasoning effort can multiply them several times over. Two
consequences:

- Explicit "think step by step" prompting is for **non-reasoning** models. On a
  reasoning model it is redundant and adds cost.
- `max_completion_tokens` must be generous on reasoning models, because hidden
  reasoning is drawn from the same budget. Set it too low and you get an empty
  answer with `finish_reason: "length"` — reasoning consumed the entire allowance
  before any visible text was produced.

## 3. Prompt engineering, measured

Four techniques on one classification task, scored against a small labelled set.

In [ ]:
CASES = [
    ("My order is 3 days late and nobody replies.", "COMPLAINT"),
    ("Do you ship to Norway?", "QUESTION"),
    ("The CX-4400 is superb, best purchase this year.", "PRAISE"),
    ("I want to cancel my subscription immediately.", "CANCELLATION"),
    ("It works fine but the manual is confusing.", "COMPLAINT"),
    ("Can I add a second seat to my plan?", "QUESTION"),
]
LABELS = ["COMPLAINT", "QUESTION", "PRAISE", "CANCELLATION"]


def score(build_messages, label):
    correct, tokens = 0, 0
    for text, truth in CASES:
        r = client.chat.completions.create(
            model=MINI, messages=build_messages(text), temperature=0, max_tokens=250)
        out = (r.choices[0].message.content or "").strip().upper()
        got = next((l for l in LABELS if l in out), "?")
        correct += got == truth
        tokens += r.usage.total_tokens
    print(f"{label:<26} {correct}/{len(CASES)} correct   {tokens:>5} tokens")
    return correct, tokens


score(lambda t: [{"role": "user", "content": f"Classify: {t}"}], "zero-shot, vague")

score(lambda t: [
    {"role": "system", "content":
     f"Classify the message as exactly one of {LABELS}. Reply with the label only."},
    {"role": "user", "content": t},
], "zero-shot, constrained")

FEW_SHOT = [
    {"role": "system", "content": f"Classify as one of {LABELS}. Reply with the label only."},
    {"role": "user", "content": "Still waiting on my refund, this is unacceptable."},
    {"role": "assistant", "content": "COMPLAINT"},
    {"role": "user", "content": "What voltage does it take?"},
    {"role": "assistant", "content": "QUESTION"},
    {"role": "user", "content": "Please close my account at the end of the month."},
    {"role": "assistant", "content": "CANCELLATION"},
]
score(lambda t: FEW_SHOT + [{"role": "user", "content": t}], "few-shot")

score(lambda t: [
    {"role": "system", "content":
     f"Classify as one of {LABELS}. First write REASONING: one sentence. "
     "Then write LABEL: <label> on its own line."},
    {"role": "user", "content": t},
], "chain of thought")

The interesting column is tokens. Chain of thought usually costs several times what
few-shot costs on a task this easy, for no accuracy gain. **Buy the cheapest
technique that reaches your accuracy target**, and measure rather than assuming.

> **Exam note.** Few-shot fixes *format and edge cases*. It does not add knowledge —
> that is grounding (unit 02.2). A question about the model not knowing a fact is
> never answered by few-shot.

## 4. Reflection and self-critique

Three shapes, increasing reliability:

1. **Self-critique** — the model reviews its own work. Cheapest, weakest.
2. **Critic model** — a different (usually stronger) model reviews it.
3. **Verifier** — a deterministic check. Most reliable, because it is not an opinion.

In [ ]:
TASK = (
    "Write a 3-sentence apology email to a customer whose CX-4400 arrived damaged. "
    "Do not promise a refund, a credit, or any specific date."
)

RUBRIC = """Score the DRAFT against these criteria:
1. Exactly three sentences.
2. Acknowledges the damage.
3. Promises NO refund, NO credit and NO specific date.
4. Professional, not grovelling."""

CRITIQUE_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "passes": {"type": "boolean"},
        "violations": {"type": "array", "items": {"type": "string"}},
        "fix_instruction": {"type": "string"},
    },
    "required": ["passes", "violations", "fix_instruction"],
}


def critique(draft, critic_model):
    r = client.chat.completions.create(
        model=critic_model,
        messages=[{"role": "user", "content": f"{RUBRIC}\n\nDRAFT:\n{draft}"}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "critique", "strict": True,
                                         "schema": CRITIQUE_SCHEMA}},
        temperature=0,
    )
    return json.loads(r.choices[0].message.content), r.usage.total_tokens


def reflect(critic_model, max_rounds=3):
    """generate -> critique -> revise, with a HARD cap. Never loop unbounded."""
    draft = ask(TASK + " Be slightly over-eager and reassuring.", temperature=0.8)
    spent = 0
    for rnd in range(1, max_rounds + 1):
        verdict, tok = critique(draft, critic_model)
        spent += tok
        print(f"  round {rnd}: passes={verdict['passes']}  violations={verdict['violations']}")
        if verdict["passes"]:
            return draft, rnd, spent
        draft = ask(
            f"TASK: {TASK}\n\nPREVIOUS DRAFT:\n{draft}\n\n"
            f"PROBLEMS: {verdict['violations']}\nFIX: {verdict['fix_instruction']}\n\n"
            "Rewrite it. Output the email only.",
            temperature=0.3,
        )
    return draft, max_rounds, spent   # hit the cap -> escalate, do not retry forever


print("SELF-CRITIQUE (same model judges itself):")
d1, r1, t1 = reflect(MINI)
print(f"  settled after {r1} round(s), {t1} critique tokens\n")
print(d1)

print("\nCRITIC MODEL (stronger model judges):")
d2, r2, t2 = reflect(BIG)
print(f"  settled after {r2} round(s), {t2} critique tokens\n")
print(d2)

In [ ]:
# A VERIFIER is a deterministic check. No opinion, no bias, no token cost.
import re

BANNED = ["refund", "credit", "reimburse", "monday", "tuesday", "wednesday",
          "thursday", "friday", "tomorrow", "next week"]


def verify(draft):
    problems = []
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", draft.strip()) if s]
    if len(sentences) != 3:
        problems.append(f"{len(sentences)} sentences, expected 3")
    lowered = draft.lower()
    for word in BANNED:
        if word in lowered:
            problems.append(f"contains banned term: {word}")
    if re.search(r"\b\d{1,2}[/-]\d{1,2}\b|\b\d+\s*(days?|weeks?)\b", lowered):
        problems.append("contains a specific timeframe")
    return problems


for label, draft in (("self-critique output", d1), ("critic-model output", d2)):
    problems = verify(draft)
    print(f"{label:<22} {'PASS' if not problems else 'FAIL ' + str(problems)}")

print("\nThe verifier costs nothing and never flatters the model. Where a rule can")
print("be expressed mechanically, prefer it over an LLM judge.")

> **Exam note.** A model critiquing its own output is biased toward approving it,
> which is why self-critique often "passes" on round 1 while a stronger critic — or
> a deterministic verifier — finds real violations. Always cap the loop: an
> uncapped reflect-and-retry on a stubborn failure is an unbounded bill.

## 5. Tracing to Application Insights

Foundry uses **OpenTelemetry** with the GenAI semantic conventions. Three steps:
get the connection string, configure the exporter, instrument the client.

> **Privacy.** Prompt and completion **content** is not recorded by default. Turning
> it on writes user content into Application Insights — a data-governance decision,
> not merely a switch.

In [ ]:
TRACING_ON = False
conn_str = cfg.get("APPLICATIONINSIGHTS_CONNECTION_STRING")

try:
    if not conn_str:
        # The project knows its own Application Insights connection.
        conn_str = project_client().telemetry.get_application_insights_connection_string()

    from azure.monitor.opentelemetry import configure_azure_monitor
    from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor

    configure_azure_monitor(connection_string=conn_str)
    OpenAIInstrumentor().instrument()

    # import os
    # os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"  # see warning

    TRACING_ON = True
    print("tracing configured -> Application Insights")
except Exception as e:
    print("tracing not configured:", type(e).__name__, str(e)[:220])
    print("\nUsual causes: no Application Insights connected to the project, or")
    print("azure-monitor-opentelemetry / opentelemetry-instrumentation-openai-v2 missing.")
    print("Section 6 works regardless - it uses local instrumentation.")

In [ ]:
if TRACING_ON:
    from opentelemetry import trace

    tracer = trace.get_tracer("ai103.unit_02_5")

    # A custom parent span turns three model calls into one traceable operation,
    # which is how you get a latency BREAKDOWN rather than one opaque number.
    with tracer.start_as_current_span("support_pipeline") as span:
        span.set_attribute("ai103.scenario", "damaged_order")
        span.set_attribute("ai103.customer_tier", "standard")

        with tracer.start_as_current_span("classify"):
            label = ask("Classify in one word: 'my controller arrived smashed'", temperature=0)
        with tracer.start_as_current_span("draft"):
            reply = ask(f"Write a two-sentence reply to a {label} about a damaged controller.")
        with tracer.start_as_current_span("verify"):
            problems = verify(reply)
            span.set_attribute("ai103.verifier_violations", len(problems))

    print("traced. Allow 2-3 minutes, then look in Foundry > Tracing.")
    print("label:", label.strip()[:60])
else:
    print("skipped - tracing not configured")

Attributes to know by name, because they are what your KQL queries filter on:

| Attribute | Holds |
|---|---|
| `gen_ai.system` | `az.ai.openai` |
| `gen_ai.request.model` | the deployment called |
| `gen_ai.usage.input_tokens` | prompt tokens |
| `gen_ai.usage.output_tokens` | completion tokens |
| `gen_ai.response.finish_reasons` | `stop`, `length`, `content_filter`, `tool_calls` |

That last one is your safety and truncation signal. A rise in `content_filter` means
a jailbreak campaign or a bad prompt deployment; a rise in `length` means users are
seeing answers cut off mid-sentence.

## 6. Token, latency, and safety analytics

The same four signals, computed locally so you can see exactly what the dashboard is
made of.

In [ ]:
PRICES = {  # USD per 1M tokens - illustrative, check the pricing page for real values
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4o": (2.50, 10.00),
    "o4-mini": (1.10, 4.40),
}
TELEMETRY = []


def instrumented(model, messages, feature="unknown", **kw):
    """One wrapper produces all four observability signals."""
    t0 = time.perf_counter()
    error = None
    try:
        r = client.chat.completions.create(model=model, messages=messages, **kw)
    except Exception as e:
        error = type(e).__name__
        TELEMETRY.append({"feature": feature, "model": model, "error": error,
                          "latency_s": time.perf_counter() - t0, "in": 0, "out": 0,
                          "finish": "error", "filtered": True, "cost_usd": 0.0})
        raise

    latency = time.perf_counter() - t0
    u = r.usage
    rate_in, rate_out = PRICES.get(model, (0.15, 0.60))
    cost = u.prompt_tokens * rate_in / 1e6 + u.completion_tokens * rate_out / 1e6
    finish = r.choices[0].finish_reason

    TELEMETRY.append({
        "feature": feature, "model": model, "error": None, "latency_s": latency,
        "in": u.prompt_tokens, "out": u.completion_tokens, "finish": finish,
        "filtered": finish == "content_filter", "cost_usd": cost,
    })
    return r


WORKLOAD = [
    ("faq", "What are your opening hours?"),
    ("faq", "Do you ship internationally?"),
    ("triage", "My controller died after 20 months at 60C, am I covered?"),
    ("triage", "Order ORD-12345 is late and I am furious."),
    ("summarise", "Summarise: " + ("The customer called about a damaged unit. " * 12)),
]
for feature, text in WORKLOAD:
    instrumented(MINI, [{"role": "user", "content": text}], feature=feature,
                 temperature=0, max_tokens=140)

print(f"{len(TELEMETRY)} calls recorded")

In [ ]:
from collections import defaultdict

print("TOKEN AND COST BY FEATURE")
print(f"{'feature':<12}{'calls':>6}{'in':>8}{'out':>8}{'cost $':>12}{'$/1k req':>11}")
by_feature = defaultdict(list)
for row in TELEMETRY:
    by_feature[row["feature"]].append(row)
for feature, rows in sorted(by_feature.items()):
    cost = sum(r["cost_usd"] for r in rows)
    print(f"{feature:<12}{len(rows):>6}{sum(r['in'] for r in rows):>8}"
          f"{sum(r['out'] for r in rows):>8}{cost:>12.6f}{cost / len(rows) * 1000:>11.2f}")

print("\nLATENCY")
lat = sorted(r["latency_s"] for r in TELEMETRY)
print(f"  p50 {statistics.median(lat):.2f}s   "
      f"p95 {lat[min(int(len(lat) * 0.95), len(lat) - 1)]:.2f}s   max {lat[-1]:.2f}s")

print("\nSAFETY AND COMPLETION SIGNALS")
finishes = defaultdict(int)
for row in TELEMETRY:
    finishes[row["finish"]] += 1
for reason, n in finishes.items():
    flag = "  <- investigate" if reason in ("content_filter", "length", "error") else ""
    print(f"  {reason:<16} {n}{flag}")

print(f"\nfiltered rate : {sum(r['filtered'] for r in TELEMETRY) / len(TELEMETRY):.1%}")
print(f"total spend   : ${sum(r['cost_usd'] for r in TELEMETRY):.6f}")

The equivalent in Application Insights, once traces flow:

```kusto
dependencies
| where timestamp > ago(24h)
| extend model  = tostring(customDimensions["gen_ai.request.model"]),
         in_tok = toint(customDimensions["gen_ai.usage.input_tokens"]),
         out_tok= toint(customDimensions["gen_ai.usage.output_tokens"]),
         reason = tostring(customDimensions["gen_ai.response.finish_reasons"])
| where isnotempty(model)
| summarize calls = count(),
            input = sum(in_tok), output = sum(out_tok),
            p95_ms = percentile(duration, 95),
            filtered = countif(reason contains "content_filter"),
            truncated = countif(reason contains "length")
          by model, bin(timestamp, 1h)
| order by timestamp desc
```

## 7. Orchestrating multiple models

### 7a. Routing — classify first, then choose the model

In [ ]:
ROUTE_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "complexity": {"type": "string", "enum": ["simple", "complex"]},
        "reason": {"type": "string"},
    },
    "required": ["complexity", "reason"],
}


def route(question):
    """A cheap classifier decides which model pays for the answer."""
    r = instrumented(
        MINI,
        [{"role": "system", "content":
          "Classify the question. 'simple' = factual lookup, greeting, single-step. "
          "'complex' = multi-step reasoning, numeric constraints, or conflicting rules."},
         {"role": "user", "content": question}],
        feature="router", temperature=0,
        response_format={"type": "json_schema",
                         "json_schema": {"name": "route", "strict": True, "schema": ROUTE_SCHEMA}},
    )
    return json.loads(r.choices[0].message.content)


QUESTIONS = [
    "What is your returns window?",
    "Do you ship to Norway?",
    "Three crates: A is twice B, C is 12kg under A, total 108kg, none over 50kg. Legal?",
    "Unit failed at 20 months in a 60C room, box opened in April. Entitlement?",
]

TELEMETRY.clear()
for q in QUESTIONS:
    decision = route(q)
    target = BIG if decision["complexity"] == "complex" else MINI
    instrumented(target, [{"role": "user", "content": q}],
                 feature=f"answer_{decision['complexity']}", temperature=0, max_tokens=250)
    print(f"  {decision['complexity']:<8} -> {target:<12} {q[:52]}")

routed_cost = sum(r["cost_usd"] for r in TELEMETRY)
print(f"\nrouted total: ${routed_cost:.6f}")

In [ ]:
# Baseline: send everything to the large model.
TELEMETRY.clear()
for q in QUESTIONS:
    instrumented(BIG, [{"role": "user", "content": q}], feature="all_big",
                 temperature=0, max_tokens=250)
big_cost = sum(r["cost_usd"] for r in TELEMETRY)

print(f"everything on {BIG}: ${big_cost:.6f}")
print(f"routed             : ${routed_cost:.6f}")
if big_cost:
    print(f"saving             : {(1 - routed_cost / big_cost):.0%}")
print("\nRouting is only worth it when the router is much cheaper than the answer")
print("and most traffic is simple. Measure the DISTRIBUTION of routing decisions:")
print("a router that never routes cheap has added a call and saved nothing.")

In [ ]:
# 7b. Cascade - try cheap, escalate only when a VERIFIER fails. No extra classifier
# call: the check is deterministic and free.
def cascade(question, verifier):
    cheap = instrumented(MINI, [{"role": "user", "content": question}],
                         feature="cascade_cheap", temperature=0, max_tokens=300)
    text = cheap.choices[0].message.content
    if verifier(text):
        return text, MINI, False
    strong = instrumented(BIG, [{"role": "user", "content": question}],
                          feature="cascade_escalated", temperature=0, max_tokens=400)
    return strong.choices[0].message.content, BIG, True


def has_verdict(text):
    """Cheap, deterministic acceptance test for the crate puzzle."""
    up = (text or "").upper()
    return ("LEGAL" in up or "ILLEGAL" in up) and any(ch.isdigit() for ch in up)


TELEMETRY.clear()
answer, used, escalated = cascade(
    "Three crates: A is twice B, C is 12kg under A, total 108kg, none may exceed 50kg. "
    "Give the weights then say LEGAL or ILLEGAL.", has_verdict)
print(f"answered by {used}, escalated={escalated}")
print(answer[:280])
print(f"\ncost: ${sum(r['cost_usd'] for r in TELEMETRY):.6f}")

In [ ]:
# 7c. Fallback - availability, not cost. Try deployments in order on 429/5xx.
def with_fallback(messages, deployments, **kw):
    errors = []
    for dep in deployments:
        try:
            return instrumented(dep, messages, feature="fallback", **kw), dep, errors
        except Exception as e:
            errors.append((dep, type(e).__name__, str(e)[:90]))
    raise RuntimeError(f"all deployments failed: {errors}")


TELEMETRY.clear()
resp, used, errs = with_fallback(
    [{"role": "user", "content": "Reply with the single word: ok"}],
    ["this-deployment-does-not-exist", MINI],   # first one 404s
    temperature=0, max_tokens=10,
)
print("failed over from:", [e[0] for e in errs])
print("answered by     :", used, "->", resp.choices[0].message.content)
print("\nIn production the fallback list is usually the same model in a second region,")
print("and you retry 429 and 5xx but NOT 400 or content_filter - those will not")
print("succeed anywhere and retrying just multiplies the cost.")

In [ ]:
# 7d. Hybrid LLM + rules engine. Deterministic rules answer what they can - free,
# instant, auditable - and the model only sees the ambiguous remainder.
RULES = [
    (re.compile(r"\b(hours?|open(ing)?|closed)\b", re.I),
     "Support is open 09:00-17:00 UTC, Monday to Friday."),
    (re.compile(r"\b(returns?|refund)\s+(window|period|policy)\b", re.I),
     "Unopened items: 30 days for a full refund. Opened items: store credit only."),
    (re.compile(r"\bORD-\d{5}\b", re.I), None),   # matched but needs a lookup -> LLM
]


def hybrid(question):
    for pattern, canned in RULES:
        if pattern.search(question):
            if canned:
                return canned, "rules", 0.0
            break
    before = sum(r["cost_usd"] for r in TELEMETRY)
    r = instrumented(MINI, [{"role": "user", "content": question}],
                     feature="hybrid_llm", temperature=0, max_tokens=160)
    return r.choices[0].message.content, "llm", sum(x["cost_usd"] for x in TELEMETRY) - before


TELEMETRY.clear()
MIX = [
    "What are your opening hours?",
    "What is the returns window?",
    "Are you open on Saturday?",
    "My CX-4400 fails intermittently above 50C, is that covered?",
    "Where is ORD-12345?",
]
handled = {"rules": 0, "llm": 0}
for q in MIX:
    text, source, cost = hybrid(q)
    handled[source] += 1
    print(f"  [{source:<5}] ${cost:.6f}  {q[:44]}")

print(f"\nrules handled {handled['rules']}/{len(MIX)} at zero cost, zero latency,")
print("fully deterministic and auditable. That is the argument for a hybrid design,")
print("and in regulated domains the deterministic path is often mandatory.")

| Pattern | Optimises | Extra cost | Watch out for |
|---|---|---|---|
| Routing | cost | one cheap classifier call | A router biased to "complex" saves nothing |
| Cascade | cost, with a quality floor | the failed cheap attempt | Escalation rate must stay low |
| Fallback | availability | only on failure | Never retry 400 or `content_filter` |
| Ensemble / best-of-n | quality | n× | Rarely worth it outside high-stakes work |
| Hybrid + rules | cost, latency, auditability | rule maintenance | Rules drift out of date silently |

## 8. Cleanup

This lab creates no agents, indexes, or deployments. It does flush telemetry — give
the exporter a moment so the traces reach Application Insights before the kernel
stops.

In [ ]:
if TRACING_ON:
    try:
        from opentelemetry import trace as _trace

        provider = _trace.get_tracer_provider()
        if hasattr(provider, "force_flush"):
            provider.force_flush()
        print("telemetry flushed - check Foundry > Tracing in a few minutes")
    except Exception as e:
        print("flush failed:", type(e).__name__)

TELEMETRY.clear()
print("nothing billable was created by this lab")
print("\nIf you have finished the course, run 99_teardown - the Azure AI Search")
print("service from unit 02.2 is still billing by the hour.")

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Starve a reasoning model.** Call `MODEL_REASONING` on the crate puzzle with
   `max_completion_tokens=200`. What comes back in `message.content`, what is the
   `finish_reason`, and what does `reasoning_tokens` say? Explain in one sentence why
   this fails differently from truncating a `gpt-4o-mini` answer.
2. **Both knobs at once.** Sample the description prompt eight times at
   `temperature=1.5, top_p=0.1`, then at `temperature=0.2, top_p=1.0`. Compare
   `distinct_ratio`. Which parameter dominated, and what does that tell you about
   setting both?
3. **Break the self-critic.** Change `reflect()` so the critic is instructed to be
   generous and encouraging. Run it, then run `verify()` on the result. How often
   does a "passing" draft fail the deterministic check, and what is the general
   lesson about LLM-as-judge?
4. **Router economics.** Build a 20-question workload that is 80% simple. Compute
   the break-even point: at what fraction of complex questions does routing stop
   saving money, given that every request now pays for a router call?

In [ ]:
# Your work here.